# Part 1: Data Preparation and Version Control with DVC

This notebook handles:
1. Loading the raw SMS Spam Collection dataset.
2. Splitting it into training, validation, and test sets.
3. Saving the splits to CSV files for DVC tracking.

### DVC Setup Instructions

Run the following commands in your terminal to initialize DVC and start tracking data:

```bash
# Initialize DVC in the project root
dvc init

# Commit the DVC setup to git
git add .dvc .dvcignore
git commit -m "Initialize DVC"
```

After running the cells below to generate data:

```bash
# Track the raw data and splits
dvc add raw_data.csv train.csv validation.csv test.csv

# Commit the DVC tracking files (.dvc) to git
git add raw_data.csv.dvc train.csv.dvc validation.csv.dvc test.csv.dvc .gitignore
git commit -m "Add data versions"
```

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
import zipfile
import io
import requests

# Initialize random seed - change this to generate a new version of the data
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Load Raw Dataset
We load the SMS Spam Collection dataset directly.

In [ ]:
# Load dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
try:
    print("Downloading dataset...")
    r = requests.get(url)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    # The file inside is 'SMSSpamCollection'
    with z.open('SMSSpamCollection') as f:
        df = pd.read_csv(f, sep='\t', header=None, names=['label', 'message'], encoding='latin-1', on_bad_lines='skip')
except Exception as e:
    print(f"Error loading data: {e}")
    # Fallback to local file if available
    if os.path.exists('SMSSpamCollection'):
        df = pd.read_csv('SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])

# Encode labels: ham=0, spam=1
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

print(f"Dataset shape: {df.shape}")
print(df.head())

# Save raw data
df.to_csv('raw_data.csv', index=False)
print("Saved raw_data.csv")

## 2. Train/Validation/Test Split
Split the data: 60% Train, 20% Validation, 20% Test.

In [ ]:
# Split: Train (60%), Temp (40%)
train_df, temp_df = train_test_split(df, test_size=0.4, random_state=RANDOM_SEED, stratify=df['label'])

# Split Temp: Validation (50% of Temp -> 20% total), Test (50% of Temp -> 20% total)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=RANDOM_SEED, stratify=temp_df['label'])

print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

# Save splits
train_df.to_csv('train.csv', index=False)
val_df.to_csv('validation.csv', index=False)
test_df.to_csv('test.csv', index=False)

print("Saved train.csv, validation.csv, test.csv")

## 3. Creating a New Data Version

To create a new version of the data splits:
1. Change `RANDOM_SEED` in the second cell (e.g., to 123).
2. Run all cells again to regenerate the CSV files.
3. Run the following DVC commands:

```bash
# Identify changed files
dvc status

# Track the new version
dvc commit

# Commit changes to git
git add raw_data.csv.dvc train.csv.dvc validation.csv.dvc test.csv.dvc
git commit -m "Update data split with new seed"
```

## 4. Switching Between Versions

To switch back to the previous version:

```bash
# Checkout the previous git commit
git checkout HEAD^

# Checkout the DVC data associated with that commit
dvc checkout
```

You can verify the data distribution has changed by reloading the CSVs:

In [ ]:
def print_distribution(name, df):
    counts = df['label'].value_counts()
    print(f"Distribution in {name}:\n{counts}\n")

# Reload to verify content on disk
try:
    t = pd.read_csv('train.csv')
    v = pd.read_csv('validation.csv')
    te = pd.read_csv('test.csv')

    print(f"--- Random Seed: {RANDOM_SEED} ---")
    print_distribution('Train', t)
    print_distribution('Validation', v)
    print_distribution('Test', te)
except FileNotFoundError:
    print("Files not found. Run the generation cells or dvc checkout.")

## 5. Bonus: Google Drive Remote Storage

To configure Google Drive as a remote storage for DVC, allowing you to decouple compute and storage:

1. Create a folder in Google Drive and get its ID (from the URL).
2. Run the following commands:

```bash
# Add Google Drive remote
dvc remote add -d myremote gdrive://<YOUR-GDRIVE-FOLDER-ID>

# Commit the config
git add .dvc/config
git commit -m "Configure Google Drive remote"

# Push data to remote
dvc push
```

When you or a colleague wants to pull the data on a new machine:

```bash
dvc pull
```

This stores the large dataset artifacts in Google Drive, keeping the git repository light.